# TXTFIRE Alert Analysis

## Background
TXTFIRE was originally an SMS-based alert system for fires in Manila and now has expanded to other social media, namely Facebook and Viber. The alert system was started in 2002 by Gerie Chua, the owner of the Eng Bee Tin hopia company that started in Binondo, Manila's Chinatown [1]. Binondo and the neighboring Divisoria and Tondo neighborhoods are densely populated commercial and residential districts where fires can spread rapidly through tightly packed urban spaces. In response to the risks of fire, communities often relied on mutual aid from volunteer firefighting brigades to protect homes and businesses to supplement official responders. In particular, the Chinese-Filipino community historically played a major role in volunteer firefighting initiatives in Metro Manila [2].



## Sources
[1] TXTFIRE Philippines Foundation Inc. (n.d.). Our history. TXTFIRE Philippines. [[History of TXTFIRE]](https://txtfire.net/our-history/)

[2] Carlo Magno. The Predisposition to Help of Filipino and Chinese-Filipino Firefighters. International Journal of Research and Review, 2010. [[ResearchGate]](https://www.researchgate.net/publication/277405294_The_Predisposition_to_Help_of_Filipino_and_Chinese-Filipino_Firefighters])

## Analysis Notes
Using Apify, I archived TXTFIRE alerts from the [TXTFIRE Philippines Facebook page](https://www.facebook.com/txtfire). This was more reliable than using a scheduled webscraper of the TXTFIRE website over time.

In [2]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv('OPENAI_API_KEY')
)

In [3]:
import pandas as pd

pd.set_option('display.max_colwidth', None)

In [5]:
txtfire_fb = pd.read_csv('003_data/dataset_facebook-posts-scraper_2026-05-13_00-18-14-149.csv')

In [7]:
txtfire_fb.columns

Index(['comments', 'facebookId', 'facebookUrl', 'feedbackId', 'inputUrl',
       'likes', 'link', 'media/0/__isMedia', 'media/0/__typename',
       'media/0/accent_color',
       ...
       'textReferences/8/work_info', 'time', 'timestamp', 'topLevelUrl',
       'topReactionsCount', 'url', 'user/id', 'user/name', 'user/profilePic',
       'user/profileUrl'],
      dtype='str', length=483)

In [8]:
txtfire_fb.shape

(999, 483)

In [12]:
txtfire_fb['timestamp'] = pd.to_datetime(txtfire_fb['timestamp'], unit='s')

In [13]:
txtfire_fb['timestamp']

0     2026-05-12 23:00:47
1     2026-05-12 22:01:19
2     2026-05-12 20:00:28
3     2026-05-12 19:13:57
4     2026-05-12 19:13:49
              ...        
994   2026-04-24 22:39:28
995   2026-04-24 22:39:19
996   2026-04-24 22:17:58
997   2026-04-24 22:17:51
998   2026-04-24 22:03:40
Name: timestamp, Length: 999, dtype: datetime64[s]

In [15]:
txtfire_fb['timestamp'].agg(['min', 'max'])

min   2026-04-24 22:03:40
max   2026-05-12 23:00:47
Name: timestamp, dtype: datetime64[s]

In [14]:
txtfire_fb['likes']

0       8
1       5
2       8
3       8
4       7
       ..
994    16
995    18
996    20
997    12
998    27
Name: likes, Length: 999, dtype: int64

In [16]:
txtfire_fb['likes'].agg(['min', 'max'])

min      4
max    249
Name: likes, dtype: int64

In [41]:
txtfire_fb['post_type'] = 'other'

txtfire_fb.loc[
    txtfire_fb['text'].str.lower().str.contains('mr ube', na=False),
    'post_type'
] = 'mr_ube'

txtfire_fb.loc[
    txtfire_fb['text'].str.lower().str.contains('weather alert', na=False),
    'post_type'
] = 'weather'

txtfire_fb.loc[
    txtfire_fb['text'].str.lower().str.contains('fuel price', na=False),
    'post_type'
] = 'fuel_price'


txtfire_fb.loc[
    txtfire_fb['text'].str.lower().str.contains('fire alert|positive alarm|visible smoke|fire out|task force', na=False),
    'post_type'
] = 'fire_alert'

In [42]:
txtfire_fb['post_type'].value_counts()

post_type
fire_alert    664
mr_ube        261
weather        60
other           8
fuel_price      6
Name: count, dtype: int64

I will remove the footers from the text.

In [48]:
txtfire_fb['text_clean'] = txtfire_fb['text'].str.split('\n\n').str[0]

In [51]:
txtfire_fb['city_hint'] = None

txtfire_fb.loc[
    txtfire_fb['text'].str.contains('– QC', na=False),
    'city_hint'
] = 'Quezon City'

In [52]:
txtfire_fb['text_clean'] = (
    txtfire_fb['text']
    .str.replace(r'^Fire Alert!\n->\s*', '', regex=True)
    .str.replace(r'^🚨 TXTFIRE Fire Alert! – [A-Z]+\n📍\s*', '', regex=True)
    .str.split('\n\n')
    .str[0]
)

I want to classify the events.

In [61]:
text_lower = txtfire_fb['text_clean'].str.lower()

txtfire_fb['fire_status'] = 'other'

txtfire_fb.loc[
    text_lower.str.contains('for verification', na=False),
    'fire_status'
] = 'for_verification'

txtfire_fb.loc[
    text_lower.str.contains('visible smoke', na=False),
    'fire_status'
] = 'visible_smoke'

txtfire_fb.loc[
    text_lower.str.contains('confined', na=False),
    'fire_status'
] = 'confined'

txtfire_fb.loc[
    text_lower.str.contains('under control', na=False),
    'fire_status'
] = 'under_control'

txtfire_fb.loc[
    text_lower.str.contains('fire out', na=False),
    'fire_status'
] = 'fire_out'

txtfire_fb.loc[
    text_lower.str.contains('alarm', na=False),
    'fire_status'
] = 'alarm'

In [62]:
txtfire_fb[txtfire_fb['post_type'] == 'fire_alert'][['timestamp', 'text_clean', 'fire_status']]

,timestamp,text_clean,fire_status
3,2026-05-12 19:13:57,maligaya street brgy pasong putik Quezon City : Fire Out,fire_out
4,2026-05-12 19:13:49,"Maligaya Street Brgy. Pasong Putik, QC – Fire Out\nUpdate as of 3:13:49 AM",fire_out
5,2026-05-12 19:04:54,maligaya street brgy pasong putik Quezon City : Fire Under Control,under_control
6,2026-05-12 19:04:46,"Maligaya Street Brgy. Pasong Putik, QC – Under Control\nUpdate as of 3:04:46 AM",under_control
7,2026-05-12 18:06:47,maligaya street brgy pasong putik Quezon City : 3rd Alarm,alarm
...,...,...,...
994,2026-04-24 22:39:28,commonwealth market barangay commonwealth Quezon City : Fire Out,fire_out
995,2026-04-24 22:39:19,"Commonwealth Market Brgy. Commonwealth, Quezon City – Fire Out\nUpdate as of 6:39:19 AM",fire_out
996,2026-04-24 22:17:58,commonwealth market barangay commonwealth Quezon City : Fire Under Control,under_control
997,2026-04-24 22:17:51,"Commonwealth Market Brgy. Commonwealth, Quezon City – Under Control\nUpdate as of 6:17:50 AM",under_control
